# HyperKT — Stage 9A: Five-seed statistical analysis and protocol lock
**Runtime: CPU (Python 3).** Inputs: `hyperkt_stage8b.zip` and `hyperkt_stage2(2).zip`. Output: `hyperkt_stage9a.zip`.

This notebook does **not** evaluate the held-out test labels. The five random seeds share the same validation students, so seed-level confidence intervals describe variation over training seeds, **not** uncertainty across independent students/datasets. No statistical significance or causal prerequisite claim is assumed.

In [ ]:
import os, io, json, zipfile, hashlib, shutil, pathlib
import numpy as np, pandas as pd
from scipy import stats
from google.colab import files
OUT='/content/hyperkt_stage9a'; os.makedirs(OUT,exist_ok=True)
print('Upload Stage 8B and Stage 2 ZIP files:')
uploaded=files.upload()
paths={k:'/content/'+k for k in uploaded}
p8=next((v for k,v in paths.items() if 'stage8b' in k.lower()),None)
p2=next((v for k,v in paths.items() if 'stage2' in k.lower()),None)
assert p8 and p2, 'Both Stage 8B and Stage 2 ZIP files are required'
assert zipfile.is_zipfile(p8) and zipfile.is_zipfile(p2)
print('Loaded:',p8,p2)

In [ ]:
with zipfile.ZipFile(p8) as z:
    names=set(z.namelist())
    required={'all_results.csv','config.json','audit.json','paired_auc_differences.csv'}
    assert required.issubset(names),f'Missing: {required-names}'
    df=pd.read_csv(io.BytesIO(z.read('all_results.csv')))
    cfg=json.loads(z.read('config.json'))
    prior_audit=json.loads(z.read('audit.json'))
print(df.to_string(index=False))
assert len(df)==20
assert not df.duplicated(['seed','variant']).any()
assert df['seed'].nunique()==5 and df['variant'].nunique()==4
assert df['n'].nunique()==1
assert df[['auc','nll','brier','f1','ece15']].notna().all().all()
print('All 20 run records present.')

In [ ]:
# SHA-256 fingerprints: check provenance rather than assuming filenames imply identical files.
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
actual=sha256(p2)
expected=cfg.get('stage2_sha256')
print('Stage 2 SHA256:',actual)
print('Expected:',expected)
assert actual==expected, 'Stage 2 ZIP differs from the one used in Stage 8B. Upload the exact Stage 2 archive.'
print('Dataset provenance verified.')

In [ ]:
# Independent recomputation of mean and sample SD from per-run metrics.
metric_cols=['auc','nll','brier','f1','ece15']
summary=df.groupby('variant')[metric_cols].agg(['mean','std']).round(9)
print(summary.to_string())
summary.to_csv(f'{OUT}/five_seed_summary.csv')
df.to_csv(f'{OUT}/all_results_verified.csv',index=False)

In [ ]:
# Paired seed differences against plain Transformer and strongest non-hypergraph model.
wide=df.pivot(index='seed',columns='variant',values='auc').sort_index()
comparisons=[('transformer_decay','transformer'),('transformer_hypergraph','transformer'),('transformer_hypergraph_decay','transformer'),('transformer_hypergraph_decay','transformer_decay')]
rows=[]
for a,b in comparisons:
    d=(wide[a]-wide[b]).to_numpy(dtype=float)
    mean=float(d.mean()); sd=float(d.std(ddof=1)); sem=sd/np.sqrt(len(d))
    tcrit=stats.t.ppf(.975,df=len(d)-1)
    ci=(mean-tcrit*sem,mean+tcrit*sem)
    p=float(stats.ttest_1samp(d,0).pvalue)
    rows.append(dict(comparison=f'{a} - {b}',n_seeds=len(d),mean_delta_auc=mean,sd_delta_auc=sd,ci95_low=ci[0],ci95_high=ci[1],paired_t_p_exploratory=p,positive_seeds=int((d>0).sum()),negative_seeds=int((d<0).sum())))
paired=pd.DataFrame(rows)
print(paired.to_string(index=False))
paired.to_csv(f'{OUT}/paired_seed_statistics.csv',index=False)
print('Caution: exploratory tests with n=5 seeds, shared validation cohort and multiple comparisons.')

In [ ]:
# Sequence integrity audit without reading test labels or test arrays.
with zipfile.ZipFile(p2) as z:
    arrays={}
    for split in ['train','val']:
        a=np.load(io.BytesIO(z.read(f'{split}_sequences.npz')))
        arrays[split]={k:a[k] for k in a.files}
        print(split, {k:v.shape for k,v in arrays[split].items()})
users_train=set(arrays['train']['user_id'].tolist())
users_val=set(arrays['val']['user_id'].tolist())
assert users_train.isdisjoint(users_val)
checks={}
for split,a in arrays.items():
    m=a['mask'].astype(bool)
    assert a['history_skill'].shape==a['target_skill'].shape==m.shape
    assert np.isin(a['target_correct'][m],[0,1]).all()
    # Within-window shift: every target at t must match history at t+1 for consecutive valid targets.
    consecutive=m[:,:-1]&m[:,1:]
    skill_equal=(a['target_skill'][:,:-1]==a['history_skill'][:,1:])
    correct_equal=(a['target_correct'][:,:-1]==a['history_correct'][:,1:])
    assert skill_equal[consecutive].all() and correct_equal[consecutive].all(),f'{split}: local shift mismatch'
    checks[split]={'windows':int(len(m)),'targets':int(m.sum()),'students_with_windows':len(set(a['user_id'].tolist())),'local_shift_pass':True}
print(checks)
print('Cross-window event uniqueness remains UNVERIFIED: original event identifiers are absent.')

In [ ]:
# Critical reproducibility check: Stage 8B archive has no trained checkpoints.
checkpoint_files=sorted(x for x in names if x.endswith(('.pt','.pth','.ckpt','.safetensors')))
print('Model checkpoint files in Stage 8B ZIP:',checkpoint_files)
if not checkpoint_files:
    print('ACTION REQUIRED: recover best checkpoints from Google Drive or rerun training before final test.')
protocol={
 'stage':'9A','primary_selection_metric':'validation_auc','candidate_variants':list(wide.columns),
 'candidate_seeds':[int(x) for x in wide.index],
 'recommended_architecture':'transformer_hypergraph_decay',
 'selection_rationale':'highest five-seed mean validation AUC among four pre-specified Stage 8B variants',
 'config':cfg['config'],'stage2_sha256':actual,'stage8b_sha256':sha256(p8),
 'validation_only':True,'test_set_evaluated':False,
 'checkpoints_in_stage8b_zip':checkpoint_files,
 'final_test_ready':False,
 'blocking_issues':['recover or retrain saved best model checkpoints','resolve cross-window event uniqueness using original event IDs or reconstruction','inspect full causal mask and padding behavior against model source','predefine final model/seed aggregation and baseline protocol'],
 'limitations':['five seeds on one validation cohort','positional attention penalty is not elapsed-time forgetting','Stage 8B architecture differs from Stage 4','structural graph associations are not causal prerequisites'],
 'student_split_checks':checks,'prior_audit':prior_audit
}
with open(f'{OUT}/protocol_lock_DRAFT.json','w') as f:json.dump(protocol,f,indent=2)
print('DRAFT protocol saved; final test is NOT unlocked.')

In [ ]:
# Export audit and results, including explicit unresolved blockers.
with open(f'{OUT}/README.txt','w') as f:
    f.write('HyperKT Stage 9A statistical audit. Validation-only; no held-out test metrics.\n')
    f.write('Protocol lock is a DRAFT. Recover checkpoints and resolve outstanding integrity checks before final testing.\n')
archive=shutil.make_archive('/content/hyperkt_stage9a','zip',OUT)
print('Created:',archive)
files.download(archive)